In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions   
#     3. Daily Theme Listing Impressions (adv search)

In [67]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [68]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

all_dates = pd.date_range(
    start=f"{year}-{str_month}-01",
    end=pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    freq="D",
)

In [81]:
template_file = 'LSLS_report_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

'LSLS_report_2026-09-25.xlsx'

In [70]:
# 1. Daily Badge impression by device     DONE
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2
# 
# #少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ 
FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()


<>:13: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_56340\1521017806.py:13: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [30]:
# df_big_query

In [71]:
pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df

platform,index,Web,Mobile Web,Android,IOS
0,2026-08-01,20257,39493,87756,440460
1,2026-08-02,17623,36689,81841,417045
2,2026-08-03,44780,32543,64903,326647
3,2026-08-04,50576,34758,73025,363109
4,2026-08-05,50139,34689,74189,368825
5,2026-08-06,51556,34699,76806,376998
6,2026-08-07,54856,37493,79905,403192
7,2026-08-08,30397,42042,90812,453735
8,2026-08-09,32759,37463,92661,460509
9,2026-08-10,55797,34911,71397,357474


In [82]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=2, header=None, index=False)

In [ ]:
# 2. Daily Brand Page (LMS) Pageview
# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/l-%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97-l35336
# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/l-%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97-l35336?sortBy=ORScoreDesc
# app: 11:25:26|| or.search.layer.search| CityID:0;geo:22.2915762%2C114.2081518;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# app: 11:25:26|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        ( (LOWER(EventAction) = 'view.sr1.promotion') and (LOWER(EventLabelRaw) LIKE '%/l-%e5%b0%91%e9%b9%bd%e5%b0%91%e7%b3%96%e9%a3%9f%e5%ba%97-l35336%') )   -- web
        OR 
        ( (LOWER(EventAction) = 'or.search.layer.search') and (LOWER(EventLabelRaw) LIKE '%35336%') )    -- app and web
    )
    group by platform, querydate
    """

df_big_query_2_2 = client.query(sql).result().to_dataframe()


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [94]:
# df_big_query_2_2

In [92]:
pivoted_df_2 = df_big_query_2_2.pivot(index='querydate', columns='platform', values='count')
pivoted_df_2 = pivoted_df_2.reindex(index=all_dates, fill_value=0).fillna(0)

expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_2 = pivoted_df_2.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_2["Web"]=pivoted_df_2["desktop"] + pivoted_df_2["mobile"]
pivoted_df_2["App"]=pivoted_df_2["ios"] + pivoted_df_2["android"] + pivoted_df_2["hms"]
pivoted_df_2 =pivoted_df_2[["Web","App"]]

pivoted_df_2.index.name = None
pivoted_df_2.reset_index(inplace=True)
pivoted_df_2["index"] = pivoted_df_2["index"].dt.date
pivoted_df_2

platform,index,Web,App
0,2026-08-01,0,0
1,2026-08-02,0,2
2,2026-08-03,0,0
3,2026-08-04,0,1
4,2026-08-05,0,0
5,2026-08-06,0,1
6,2026-08-07,0,0
7,2026-08-08,0,0
8,2026-08-09,0,5
9,2026-08-10,0,1


In [84]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_2.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=37, header=None, index=False)

In [ ]:
# 3. Daily Theme Listing Impressions (adv search)

# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/restaurants/type/%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97?sortBy=ORScoreDesc
# app: 11:26:28|| or.search.adv| CityID:0;geo:22.2915762%2C114.2081518;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.AdvSearch
# app: 11:26:28|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        ((LOWER(EventAction) = 'view.sr1.promotion') and  (LOWER(EventLabelRaw) LIKE '%/restaurants/type/%e5%b0%91%e9%b9%bd%e5%b0%91%e7%b3%96%e9%a3%9f%e5%ba%97%') )  -- web
        OR
        ((LOWER(EventAction) = 'or.search.adv') and (LOWER(EventLabelRaw) LIKE '%35336%'))   -- app
    )
    group by platform, querydate
    """

df_big_query_3 = client.query(sql).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
#df_big_query_3

In [88]:
pivoted_df_3 = df_big_query_3.pivot(index='querydate', columns='platform', values='count')
pivoted_df_3 = pivoted_df_3.reindex(index=all_dates, fill_value=0).fillna(0)
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_3 = pivoted_df_3.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_3["Web"]= pivoted_df_3["desktop"] + pivoted_df_3["mobile"]
pivoted_df_3["App"]=pivoted_df_3["ios"] + pivoted_df_3["android"] + pivoted_df_3["hms"]
pivoted_df_3 =pivoted_df_3[["Web","App"]]


pivoted_df_3.index.name = None
pivoted_df_3.reset_index(inplace=True)
pivoted_df_3["index"] = pivoted_df_3["index"].dt.date
pivoted_df_3

platform,index,Web,App
0,2026-08-01,0,1
1,2026-08-02,0,0
2,2026-08-03,0,4
3,2026-08-04,0,3
4,2026-08-05,0,2
5,2026-08-06,0,2
6,2026-08-07,0,1
7,2026-08-08,0,1
8,2026-08-09,0,9
9,2026-08-10,1,1


In [86]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_3.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=72, header=None, index=False)